<a href="https://colab.research.google.com/github/ayaaawad/fyp-web-page/blob/main/Vein.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== ONE FULL COLAB CELL (SAME AS YOURS) + AUTO SAVE/RESUME FROM GOOGLE DRIVE =====
# It will: resnet18
# 1) Mount Drive
# 2) If checkpoints exist in Drive -> copy to runtime and resume
# 3) Train + evaluate EER
# 4) After EVERY epoch -> save last/best checkpoints to Drive (so you never lose them)

!pip -q install datasets

import os, re, random, shutil
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from tqdm import tqdm
from collections import defaultdict

# -----------------------
# Mount Google Drive + checkpoint folder
# -----------------------
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/finger_vein_checkpoints"
os.makedirs(DRIVE_DIR, exist_ok=True)
DRIVE_LAST = os.path.join(DRIVE_DIR, "last_checkpoint.pt")
DRIVE_BEST = os.path.join(DRIVE_DIR, "best_checkpoint.pt")

# If runtime is fresh but Drive has checkpoints, restore them
if os.path.exists(DRIVE_LAST) and not os.path.exists("last_checkpoint.pt"):
    shutil.copy(DRIVE_LAST, "last_checkpoint.pt")
    print("✅ Restored last_checkpoint.pt from Drive")
if os.path.exists(DRIVE_BEST) and not os.path.exists("best_checkpoint.pt"):
    shutil.copy(DRIVE_BEST, "best_checkpoint.pt")
    print("✅ Restored best_checkpoint.pt from Drive")

# -----------------------
# Settings (tuned to improve stability/accuracy)
# -----------------------
seed = 42
epochs = 10              # train longer
batch_size = 64
lr = 3e-4                # lower LR for better stability
emb_dim = 256
img_size = 224
max_pairs = 80000        # pairs for EER sampling
use_preprocess = True    # your preprocessing on
num_workers = 0          # safest with zip-backed images (prevents worker decode issues)

random.seed(seed); np.random.seed(seed)
torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -----------------------
# Load dataset + decode=False to access paths
# -----------------------
ds = load_dataset("luyu0311/MMCBNU_6000", split="train")
label_names = ds.features["label"].names
ds_nd = ds.cast_column("image", Image(decode=False))

# -----------------------
# Build physical-finger identity: subject_id × finger_type from path
# Example path contains: ".../Captured images/001/L_Fore/01.bmp..."
# -----------------------
def parse_subject_and_finger(path: str):
    m = re.search(r"Captured images/(\d{3})/([^/]+)/", path)
    if not m:
        return None, None
    return m.group(1), m.group(2)

def add_finger_key(ex):
    path = ex["image"]["path"]
    subject, finger = parse_subject_and_finger(path)
    if subject is None:
        finger = label_names[ex["label"]]
        return {"finger_key": f"UNKNOWN_{finger}"}
    return {"finger_key": f"{subject}_{finger}"}

ds2 = ds_nd.map(add_finger_key)
uniq_keys = sorted(set(ds2["finger_key"]))
print("Unique finger identities (subject×finger):", len(uniq_keys))
print("First 10 keys:", uniq_keys[:10])
assert len(uniq_keys) > 50, "Too few identities; parsing failed."

key2id = {k:i for i,k in enumerate(uniq_keys)}
ds3 = ds2.map(lambda ex: {"finger_id": key2id[ex["finger_key"]]})

# IMPORTANT: switch back to decode=True so 'image' becomes a PIL image
ds3 = ds3.cast_column("image", Image(decode=True))

# -----------------------
# Preprocessing (your pipeline)
# -----------------------
def crop_largest_contour(img):
    blur = cv2.GaussianBlur(img, (5,5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((15,15), np.uint8)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return img
    cnt = max(contours, key=cv2.contourArea)
    x,y,w,h = cv2.boundingRect(cnt)

    H, W = img.shape
    area = w*h
    if area < 0.10*H*W or area > 0.98*H*W:
        s = int(min(H,W)*0.85)
        y1 = (H - s)//2; x1 = (W - s)//2
        return img[y1:y1+s, x1:x1+s]

    margin = 10
    x1 = max(0, x - margin)
    y1 = max(0, y - margin)
    x2 = min(W, x + w + margin)
    y2 = min(H, y + h + margin)
    return img[y1:y2, x1:x2]

def apply_clahe(img, clipLimit=2.0, tileGridSize=(8,8)):
    clahe = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=tileGridSize)
    return clahe.apply(img)

def resize_with_pad(img, size=224):
    h, w = img.shape
    scale = size / max(h, w)
    new_w = max(1, int(w*scale))
    new_h = max(1, int(h*scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad = np.zeros((size, size), dtype=np.uint8)
    y = (size - new_h) // 2
    x = (size - new_w) // 2
    pad[y:y+new_h, x:x+new_w] = resized
    return pad

def preprocess(pil_img, size=224):
    img = np.array(pil_img).astype(np.uint8)
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    if use_preprocess:
        img = crop_largest_contour(img)
        img = cv2.medianBlur(img, 3)
        img = apply_clahe(img, clipLimit=2.0, tileGridSize=(8,8))
    img = resize_with_pad(img, size)
    img = img.astype(np.float32) / 255.0
    return img

# -----------------------
# Split by images per identity (same identities; different samples)
# -----------------------
finger_ids = np.array(ds3["finger_id"])
idx_by_id = defaultdict(list)
for i, fid in enumerate(finger_ids):
    idx_by_id[int(fid)].append(i)

train_idx, val_idx = [], []
rng = np.random.default_rng(seed)

for fid, idxs in idx_by_id.items():
    idxs = np.array(idxs)
    rng.shuffle(idxs)
    cut = max(1, int(0.8 * len(idxs)))
    train_idx.extend(idxs[:cut].tolist())
    val_idx.extend(idxs[cut:].tolist())

print("Train samples:", len(train_idx), "Val samples:", len(val_idx))

# -----------------------
# Torch datasets
# -----------------------
class VeinDS(Dataset):
    def __init__(self, hf_ds, indices):
        self.ds = hf_ds
        self.idx = indices
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        item = self.ds[int(self.idx[i])]
        x = preprocess(item["image"], size=img_size)
        x = torch.from_numpy(x).unsqueeze(0)  # (1,H,W)
        y = int(item["finger_id"])
        return x, y

train_loader = DataLoader(VeinDS(ds3, train_idx), batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(VeinDS(ds3, val_idx),   batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)

num_ids = len(uniq_keys)

# -----------------------
# Model
# -----------------------
class EmbedNet(nn.Module):
    def __init__(self, num_classes, emb_dim=256):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        w = base.conv1.weight.data
        base.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        base.conv1.weight.data = w.mean(dim=1, keepdim=True)

        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.emb = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, emb_dim),
            nn.BatchNorm1d(emb_dim),
        )
        self.cls = nn.Linear(emb_dim, num_classes)

    def forward(self, x):
        z = self.backbone(x)
        e = self.emb(z)
        e = F.normalize(e, dim=1)
        logits = self.cls(e)
        return logits, e

model = EmbedNet(num_classes=num_ids, emb_dim=emb_dim).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)
criterion = nn.CrossEntropyLoss()

# -----------------------
# Resume (no progress loss)
# -----------------------
start_epoch = 1
best_eer = 1e9
if os.path.exists("last_checkpoint.pt"):
    ckpt = torch.load("last_checkpoint.pt", map_location=device)
    model.load_state_dict(ckpt["model"])
    opt.load_state_dict(ckpt["opt"])
    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch = ckpt["epoch"] + 1
    best_eer = ckpt.get("best_eer", 1e9)
    print(f"✅ Resumed from epoch {ckpt['epoch']} | best_eer={best_eer*100:.2f}% | LR={opt.param_groups[0]['lr']}")
else:
    print("Starting fresh training.")

# -----------------------
# EER helpers
# -----------------------
def compute_eer(scores, labels01):
    order = np.argsort(-scores)
    scores = scores[order]
    labels01 = labels01[order]
    P = labels01.sum()
    N = len(labels01) - P
    tp = fp = 0
    fn = P
    tn = N
    best = 1e9
    eer = None
    thr = None
    for i in range(len(scores)):
        if labels01[i] == 1:
            tp += 1; fn -= 1
        else:
            fp += 1; tn -= 1
        far = fp / (fp + tn + 1e-12)
        frr = fn / (fn + tp + 1e-12)
        d = abs(far - frr)
        if d < best:
            best = d
            eer = (far + frr) / 2
            thr = scores[i]
    return float(eer), float(thr)

@torch.no_grad()
def extract_embeddings(loader):
    model.eval()
    embs, labs = [], []
    for x,y in tqdm(loader, desc="Embed", leave=False):
        x = x.to(device, non_blocking=True)
        _, e = model(x)
        embs.append(e.cpu())
        labs.append(y)
    return torch.cat(embs, 0).numpy(), torch.cat(labs, 0).numpy()

def verification_eer(embs, labs, max_pairs=80000):
    rng = np.random.default_rng(seed)
    idx_by = defaultdict(list)
    for i, lb in enumerate(labs):
        idx_by[int(lb)].append(i)
    ids = list(idx_by.keys())

    scores = []
    labels01 = []

    # genuine
    for lb, idxs in idx_by.items():
        if len(idxs) < 2:
            continue
        k = min(30, len(idxs)*(len(idxs)-1)//2)
        for _ in range(k):
            a,b = rng.choice(idxs, 2, replace=False)
            scores.append(float(np.dot(embs[a], embs[b])))
            labels01.append(1)
            if len(labels01) >= max_pairs//2:
                break
        if len(labels01) >= max_pairs//2:
            break

    # impostor
    while len(labels01) < max_pairs:
        la, lb = rng.choice(ids, 2, replace=False)
        a = rng.choice(idx_by[la])
        b = rng.choice(idx_by[lb])
        scores.append(float(np.dot(embs[a], embs[b])))
        labels01.append(0)

    return compute_eer(np.array(scores, np.float32), np.array(labels01, np.int32))

# -----------------------
# Train + Eval + Save checkpoints (+ copy to Drive every epoch)
# -----------------------
for ep in range(start_epoch, epochs+1):
    model.train()
    total_loss, total, correct = 0.0, 0, 0

    for x,y in tqdm(train_loader, desc=f"Train {ep}/{epochs}", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        logits, _ = model(x)
        loss = criterion(logits, y)
        loss.backward()
        opt.step()

        total_loss += loss.item() * x.size(0)
        total += x.size(0)
        correct += (logits.argmax(1) == y).sum().item()

    tr_loss = total_loss / total
    tr_acc = correct / total

    embs, labs = extract_embeddings(val_loader)
    eer, thr = verification_eer(embs, labs, max_pairs=max_pairs)

    print(f"Epoch {ep}/{epochs} | train loss={tr_loss:.4f} acc={tr_acc*100:.2f}% | Val EER={eer*100:.2f}% @ thr={thr:.4f}")

    # Step LR
    scheduler.step()
    print("  LR now:", opt.param_groups[0]["lr"])

    # Save LAST every epoch
    save_obj = {
        "epoch": ep,
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_eer": best_eer,
        "settings": {
            "epochs": epochs, "batch_size": batch_size, "lr": lr,
            "emb_dim": emb_dim, "img_size": img_size, "use_preprocess": use_preprocess
        }
    }
    torch.save(save_obj, "last_checkpoint.pt")
    shutil.copy("last_checkpoint.pt", DRIVE_LAST)
    print("  💾 saved last_checkpoint.pt (and copied to Drive)")

    # Save BEST by EER
    if eer < best_eer:
        best_eer = eer
        save_obj["best_eer"] = best_eer
        torch.save(save_obj, "best_checkpoint.pt")
        shutil.copy("best_checkpoint.pt", DRIVE_BEST)
        print(f"  ⭐ saved best_checkpoint.pt (and copied to Drive) new best EER = {best_eer*100:.2f}%")

print("Done. Best EER so far:", best_eer*100, "%")
print("Drive folder:", DRIVE_DIR)

Mounted at /content/drive
Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

MMCBNU_6000.zip:   0%|          | 0.00/677M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12000 [00:00<?, ? examples/s]

...

Epoch 6/10 | train loss=3.8821 acc=98.21% | Val EER=0.38% @ thr=0.4383
  LR now: 0.00015
  💾 saved last_checkpoint.pt (and copied to Drive)
  ⭐ saved best_checkpoint.pt (and copied to Drive) new best EER = 0.38%


Epoch 7/10 | train loss=3.6934 acc=99.18% | Val EER=0.38% @ thr=0.3920
  LR now: 0.00015
  💾 saved last_checkpoint.pt (and copied to Drive)


Epoch 8/10 | train loss=3.5289 acc=99.26% | Val EER=0.38% @ thr=0.3497
  LR now: 0.00015
  💾 saved last_checkpoint.pt (and copied to Drive)
  ⭐ saved best_checkpoint.pt (and copied to Drive) new best EER = 0.38%


Epoch 9/10 | train loss=3.3800 acc=99.55% | Val EER=0.26% @ thr=0.3516
  LR now: 0.00015
  💾 saved last_checkpoint.pt (and copied to Drive)
  ⭐ saved best_checkpoint.pt (and copied to Drive) new best EER = 0.26%


Train 10/10:  48%|████▊     | 72/150 [13:26<14:21, 11.05s/it]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls -lh "/content/drive/MyDrive/finger_vein_checkpoints"


Mounted at /content/drive
total 263M
-rw------- 1 root root 132M Feb 18 22:40 best_checkpoint.pt
-rw------- 1 root root 132M Feb 18 22:40 last_checkpoint.pt


In [ ]:
# ===== TEST / DEMO CELL: load saved best model (from Drive), preprocess ANY input image,
# print embedding vector clearly, and compare 2 images with cosine similarity =====

import os
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from PIL import Image
from google.colab import drive, files

# -----------------------
# 1) Mount Drive + load checkpoint
# -----------------------
drive.mount("/content/drive")

CKPT_PATH = "/content/drive/MyDrive/finger_vein_checkpoints/best_checkpoint18.pt"
assert os.path.exists(CKPT_PATH), f"Checkpoint not found at: {CKPT_PATH}"

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(CKPT_PATH, map_location=device)

# infer num_classes from checkpoint
num_classes = ckpt["model"]["cls.weight"].shape[0]
emb_dim = ckpt.get("settings", {}).get("emb_dim", 256)
img_size = ckpt.get("settings", {}).get("img_size", 224)

print("Device:", device)
print("Loaded checkpoint epoch:", ckpt.get("epoch", "?"))
print("Best saved EER:", ckpt.get("best_eer", None) * 100 if ckpt.get("best_eer", None) is not None else "unknown", "%")
print("num_classes:", num_classes, "| emb_dim:", emb_dim, "| img_size:", img_size)

# -----------------------
# 2) Model definition (must match training)
# -----------------------
class EmbedNet(nn.Module):
    def __init__(self, num_classes, emb_dim=256):
        super().__init__()
        base = models.resnet18(weights=None)

        # 1-channel input conv
        w = base.conv1.weight.data
        base.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        if w is not None and w.numel() > 0:
            base.conv1.weight.data = w.mean(dim=1, keepdim=True)

        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.emb = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, emb_dim),
            nn.BatchNorm1d(emb_dim),
        )
        self.cls = nn.Linear(emb_dim, num_classes)

    def forward(self, x):
        z = self.backbone(x)
        e = self.emb(z)
        e = F.normalize(e, dim=1)
        logits = self.cls(e)
        return logits, e

model = EmbedNet(num_classes=num_classes, emb_dim=emb_dim).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

# -----------------------
# 3) SAME preprocessing as training
# -----------------------
def crop_largest_contour(img):
    blur = cv2.GaussianBlur(img, (5,5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((15,15), np.uint8)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return img
    cnt = max(contours, key=cv2.contourArea)
    x,y,w,h = cv2.boundingRect(cnt)

    H, W = img.shape
    area = w*h
    if area < 0.10*H*W or area > 0.98*H*W:
        s = int(min(H,W)*0.85)
        y1 = (H - s)//2; x1 = (W - s)//2
        return img[y1:y1+s, x1:x1+s]

    margin = 10
    x1 = max(0, x - margin)
    y1 = max(0, y - margin)
    x2 = min(W, x + w + margin)
    y2 = min(H, y + h + margin)
    return img[y1:y2, x1:x2]

def apply_clahe(img, clipLimit=2.0, tileGridSize=(8,8)):
    clahe = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=tileGridSize)
    return clahe.apply(img)

def resize_with_pad(img, size=224):
    h, w = img.shape
    scale = size / max(h, w)
    new_w = max(1, int(w*scale))
    new_h = max(1, int(h*scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad = np.zeros((size, size), dtype=np.uint8)
    y = (size - new_h) // 2
    x = (size - new_w) // 2
    pad[y:y+new_h, x:x+new_w] = resized
    return pad

def preprocess_image(pil_img, size=224):
    img = np.array(pil_img).astype(np.uint8)
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    img = crop_largest_contour(img)
    img = cv2.medianBlur(img, 3)
    img = apply_clahe(img, clipLimit=2.0, tileGridSize=(8,8))
    img = resize_with_pad(img, size)

    img = img.astype(np.float32) / 255.0
    x = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
    return x

# -----------------------
# 4) Turn an image into an embedding vector + print it clearly
# -----------------------
@torch.no_grad()
def image_to_embedding(path_or_pil):
    if isinstance(path_or_pil, str):
        pil_img = Image.open(path_or_pil)
    else:
        pil_img = path_or_pil
    x = preprocess_image(pil_img, size=img_size).to(device)
    _, e = model(x)               # (1, emb_dim)
    e = e.squeeze(0).detach().cpu().numpy()  # (emb_dim,)
    return e

def print_embedding_info(name, emb):
    print(f"\n--- {name} embedding ---")
    print("Shape:", emb.shape)               # (256,) typically
    print("Dtype:", emb.dtype)
    print("First 20 values:", np.array2string(emb[:20], precision=4, separator=", "))
    print("Min/Max:", float(emb.min()), float(emb.max()))
    print("L2 norm (should be ~1.0):", float(np.linalg.norm(emb)))

# -----------------------
# 5) Upload 2 images and compare (self-verification demo)
# -----------------------
print("\nUpload TWO finger-vein images to compare (bmp/png/jpg).")
uploaded = files.upload()
paths = list(uploaded.keys())
assert len(paths) >= 2, "Please upload at least 2 images."

img1_path, img2_path = paths[0], paths[1]

emb1 = image_to_embedding(img1_path)
emb2 = image_to_embedding(img2_path)

print_embedding_info("Image 1", emb1)
print_embedding_info("Image 2", emb2)

cos_sim = float(np.dot(emb1, emb2))  # cosine similarity because embeddings are normalized
print("\nCosine similarity:", round(cos_sim, 6))

# -----------------------
# 6) Simple decision using the saved threshold (optional)
# Note: thresholds can vary; you can use the best epoch threshold you saw in training logs.
# If you want, set THR manually to your best epoch's thr value (e.g. 0.3516).
# -----------------------
THR = 0.35  # <-- change this to your chosen threshold from training output
decision = "MATCH (same finger)" if cos_sim >= THR else "NO MATCH (different finger)"
print("Decision @ THR =", THR, "=>", decision)

# -----------------------
# 7) (Optional) Save embedding to a file (ready to store in DB)
# -----------------------
np.save("embedding_image1.npy", emb1)
np.save("embedding_image2.npy", emb2)
print("\nSaved embeddings as embedding_image1.npy and embedding_image2.npy (these are your vectors to store/encrypt).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Loaded checkpoint epoch: 9
Best saved EER: 0.2563293392629417 %
num_classes: 606 | emb_dim: 256 | img_size: 224

Upload TWO finger-vein images to compare (bmp/png/jpg).


KeyboardInterrupt: 

In [ ]:
# ===== ENROLL (5 images) + VERIFY (1 image) DEMO CELL =====
# Flow:
# 1) Upload 5 enrollment images -> compute embeddings -> save "template" (mean embedding) + all embeddings
# 2) Upload 1 probe image -> compute embedding -> compare to template + best-of-5
# 3) Decide MATCH / NO MATCH using a threshold
#
# NOTE: Use enrollment + probe images from the SAME finger to see a MATCH.
# Use a different finger to see NO MATCH.

import os
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from PIL import Image
from google.colab import drive, files

# -----------------------
# 1) Mount Drive + load checkpoint
# -----------------------
drive.mount("/content/drive")

CKPT_PATH = "/content/drive/MyDrive/finger_vein_checkpoints/best_checkpoint18.pt"
assert os.path.exists(CKPT_PATH), f"Checkpoint not found at: {CKPT_PATH}"

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(CKPT_PATH, map_location=device)

num_classes = ckpt["model"]["cls.weight"].shape[0]
emb_dim = ckpt.get("settings", {}).get("emb_dim", 256)
img_size = ckpt.get("settings", {}).get("img_size", 224)

print("Device:", device)
print("Loaded checkpoint epoch:", ckpt.get("epoch", "?"))
print("Best saved EER:", (ckpt.get("best_eer", None) * 100) if ckpt.get("best_eer", None) is not None else "unknown", "%")
print("num_classes:", num_classes, "| emb_dim:", emb_dim, "| img_size:", img_size)

# -----------------------
# 2) Model definition (must match training)
# -----------------------
class EmbedNet(nn.Module):
    def __init__(self, num_classes, emb_dim=256):
        super().__init__()
        base = models.resnet18(weights=None)

        # 1-channel input conv
        w = base.conv1.weight.data
        base.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        if w is not None and w.numel() > 0:
            base.conv1.weight.data = w.mean(dim=1, keepdim=True)

        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.emb = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, emb_dim),
            nn.BatchNorm1d(emb_dim),
        )
        self.cls = nn.Linear(emb_dim, num_classes)

    def forward(self, x):
        z = self.backbone(x)
        e = self.emb(z)
        e = F.normalize(e, dim=1)
        logits = self.cls(e)
        return logits, e

model = EmbedNet(num_classes=num_classes, emb_dim=emb_dim).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

# -----------------------
# 3) SAME preprocessing as training
# -----------------------
def crop_largest_contour(img):
    blur = cv2.GaussianBlur(img, (5,5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((15,15), np.uint8)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return img
    cnt = max(contours, key=cv2.contourArea)
    x,y,w,h = cv2.boundingRect(cnt)

    H, W = img.shape
    area = w*h
    if area < 0.10*H*W or area > 0.98*H*W:
        s = int(min(H,W)*0.85)
        y1 = (H - s)//2; x1 = (W - s)//2
        return img[y1:y1+s, x1:x1+s]

    margin = 10
    x1 = max(0, x - margin)
    y1 = max(0, y - margin)
    x2 = min(W, x + w + margin)
    y2 = min(H, y + h + margin)
    return img[y1:y2, x1:x2]

def apply_clahe(img, clipLimit=2.0, tileGridSize=(8,8)):
    clahe = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=tileGridSize)
    return clahe.apply(img)

def resize_with_pad(img, size=224):
    h, w = img.shape
    scale = size / max(h, w)
    new_w = max(1, int(w*scale))
    new_h = max(1, int(h*scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad = np.zeros((size, size), dtype=np.uint8)
    y = (size - new_h) // 2
    x = (size - new_w) // 2
    pad[y:y+new_h, x:x+new_w] = resized
    return pad

def preprocess_image(pil_img, size=224):
    img = np.array(pil_img).astype(np.uint8)
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    img = crop_largest_contour(img)
    img = cv2.medianBlur(img, 3)
    img = apply_clahe(img, clipLimit=2.0, tileGridSize=(8,8))
    img = resize_with_pad(img, size)

    img = img.astype(np.float32) / 255.0
    x = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
    return x

# -----------------------
# 4) Turn an image into embedding
# -----------------------
@torch.no_grad()
def image_to_embedding(path):
    pil_img = Image.open(path)
    x = preprocess_image(pil_img, size=img_size).to(device)
    _, e = model(x)                     # (1, emb_dim)
    e = e.squeeze(0).detach().cpu().numpy()  # (emb_dim,)
    return e

def cosine_sim(a, b):
    # embeddings are already L2-normalized, so dot = cosine
    return float(np.dot(a, b))

# -----------------------
# 5) ENROLL: Upload 5 images
# -----------------------
print("\nSTEP 1/2: Upload exactly 5 ENROLLMENT images (same finger).")
uploaded = files.upload()
enroll_paths = list(uploaded.keys())
assert len(enroll_paths) == 5, f"Please upload EXACTLY 5 enrollment images (you uploaded {len(enroll_paths)})."

enroll_embs = np.stack([image_to_embedding(p) for p in enroll_paths], axis=0)  # (5, emb_dim)

# Template options:
# - mean embedding + renormalize (common for biometrics)
template = enroll_embs.mean(axis=0)
template = template / (np.linalg.norm(template) + 1e-12)

print("\nEnrollment done.")
print("Enrollment embeddings shape:", enroll_embs.shape)
print("Template shape:", template.shape)
print("Template first 10 values:", np.array2string(template[:10], precision=4, separator=", "))

# Save "database" locally (demo)
np.save("enroll_embeddings.npy", enroll_embs)
np.save("template_embedding.npy", template)
print("Saved: enroll_embeddings.npy, template_embedding.npy (these are what you'd encrypt/store)")

# -----------------------
# 6) VERIFY: Upload 1 probe image
# -----------------------
print("\nSTEP 2/2: Upload 1 PROBE image to VERIFY.")
uploaded2 = files.upload()
probe_paths = list(uploaded2.keys())
assert len(probe_paths) == 1, f"Please upload EXACTLY 1 probe image (you uploaded {len(probe_paths)})."
probe_path = probe_paths[0]

probe_emb = image_to_embedding(probe_path)

# Compare probe to template
sim_template = cosine_sim(probe_emb, template)

# Also compare probe to each of the 5 enrolled embeddings and take the best (more robust)
sims_each = [cosine_sim(probe_emb, enroll_embs[i]) for i in range(enroll_embs.shape[0])]
sim_best = float(np.max(sims_each))

print("\n--- Verification Scores ---")
print("Similarity vs TEMPLATE (mean):", round(sim_template, 6))
print("Similarity vs EACH enroll image:", [round(s, 6) for s in sims_each])
print("BEST similarity vs enroll set:", round(sim_best, 6))

# -----------------------
# 7) Decision threshold
# -----------------------
# Use a conservative starting threshold. You can tune later.
# If you want fewer false accepts -> increase THR.
# If you want fewer false rejects -> decrease THR.
THR = 0.35

decision_template = "MATCH" if sim_template >= THR else "NO MATCH"
decision_best = "MATCH" if sim_best >= THR else "NO MATCH"

print("\n--- Decision ---")
print("Threshold:", THR)
print("Decision using TEMPLATE:", decision_template)
print("Decision using BEST-of-5:", decision_best)

Mounted at /content/drive
Device: cuda
Loaded checkpoint epoch: 9
Best saved EER: 0.2563293392629417 %
num_classes: 606 | emb_dim: 256 | img_size: 224

STEP 1/2: Upload exactly 5 ENROLLMENT images (same finger).


Saving 01.jpg to 01.jpg
Saving 02.jpg to 02.jpg
Saving 03.jpg to 03.jpg
Saving 04.jpg to 04.jpg
Saving 05.jpg to 05.jpg

Enrollment done.
Enrollment embeddings shape: (5, 256)
Template shape: (256,)
Template first 10 values: [-0.0474,  0.0728,  0.0497, -0.0426, -0.0355,  0.1544,  0.0203, -0.0453,
  0.0432,  0.0083]
Saved: enroll_embeddings.npy, template_embedding.npy (these are what you'd encrypt/store)

STEP 2/2: Upload 1 PROBE image to VERIFY.


Saving 06.jpg to 06.jpg

--- Verification Scores ---
Similarity vs TEMPLATE (mean): 0.981594
Similarity vs EACH enroll image: [0.963969, 0.90971, 0.961377, 0.981682, 0.928489]
BEST similarity vs enroll set: 0.981682

--- Decision ---
Threshold: 0.35
Decision using TEMPLATE: MATCH
Decision using BEST-of-5: MATCH


In [ ]:
from google.colab import files
import numpy as np

THR = 0.80  # start stricter for phone images; we'll also compute a suggested threshold

def enroll_person(name, n=5):
    print(f"\nENROLL {name}: Upload exactly {n} images")
    up = files.upload()
    paths = list(up.keys())
    assert len(paths) == n, f"Upload exactly {n} images for {name}"
    embs = np.stack([image_to_embedding(p) for p in paths], axis=0)
    templ = embs.mean(axis=0)
    templ = templ / (np.linalg.norm(templ) + 1e-12)
    return embs, templ

# 1) Enroll two people
A_embs, A_templ = enroll_person("Person A", 5)
B_embs, B_templ = enroll_person("Person B", 5)

# 2) Compute impostor scores between A and B (this tells you what's "too similar")
impostor_scores = []
for i in range(5):
    impostor_scores.append(float(np.dot(A_embs[i], B_templ)))
    impostor_scores.append(float(np.dot(B_embs[i], A_templ)))

imp_mean = float(np.mean(impostor_scores))
imp_max  = float(np.max(impostor_scores))
suggested_thr = min(0.95, imp_max + 0.05)  # simple safe rule: threshold above max impostor by margin

print("\n--- Impostor stats (A vs B) ---")
print("Impostor mean:", round(imp_mean, 4))
print("Impostor max :", round(imp_max, 4))
print("Suggested THR:", round(suggested_thr, 4), "(use this if you want to be strict)")

# 3) Verify: upload 1 probe
print("\nVERIFY: Upload 1 probe image")
up = files.upload()
probe_path = list(up.keys())[0]
probe = image_to_embedding(probe_path)

sim_A = float(np.dot(probe, A_templ))
sim_B = float(np.dot(probe, B_templ))

print("\n--- Probe similarities ---")
print("Similarity to A:", round(sim_A, 6))
print("Similarity to B:", round(sim_B, 6))

# Use chosen threshold (pick either THR or suggested_thr)
USE_THR = suggested_thr  # or set USE_THR = THR
print("Using THR:", round(USE_THR, 4))

if sim_A >= USE_THR and sim_A > sim_B:
    print("Decision: MATCH Person A")
elif sim_B >= USE_THR and sim_B > sim_A:
    print("Decision: MATCH Person B")
else:
    print("Decision: NO MATCH (reject)")


ENROLL Person A: Upload exactly 5 images


Saving 02.jpg to 02.jpg
Saving 03.jpg to 03.jpg
Saving 04.jpg to 04.jpg
Saving 05.jpg to 05.jpg
Saving 06.jpg to 06.jpg


NameError: name 'image_to_embedding' is not defined

In [ ]:
# ===== ONE FULL COLAB CELL: ResNet34 (instead of ResNet18) + SAME pipeline + Save/Resume to Drive =====
!pip -q install datasets

import os, re, random, shutil
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from tqdm import tqdm
from collections import defaultdict
from google.colab import drive

# -----------------------
# Mount Drive + checkpoint folder
# -----------------------
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/finger_vein_checkpoints_resnet34"
os.makedirs(DRIVE_DIR, exist_ok=True)
DRIVE_LAST = os.path.join(DRIVE_DIR, "last_checkpoint.pt")
DRIVE_BEST = os.path.join(DRIVE_DIR, "best_checkpoint.pt")

# restore checkpoints if they exist
if os.path.exists(DRIVE_LAST) and not os.path.exists("last_checkpoint.pt"):
    shutil.copy(DRIVE_LAST, "last_checkpoint.pt")
    print("✅ Restored last_checkpoint.pt from Drive")
if os.path.exists(DRIVE_BEST) and not os.path.exists("best_checkpoint.pt"):
    shutil.copy(DRIVE_BEST, "best_checkpoint.pt")
    print("✅ Restored best_checkpoint.pt from Drive")

# -----------------------
# Settings
# -----------------------
seed = 42
epochs = 10
batch_size = 64
lr = 3e-4
emb_dim = 256
img_size = 224
max_pairs = 40000        # slightly lower to save time; increase if you want
use_preprocess = True
num_workers = 0

random.seed(seed); np.random.seed(seed)
torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -----------------------
# Load dataset + decode=False to access paths
# -----------------------
ds = load_dataset("luyu0311/MMCBNU_6000", split="train")
label_names = ds.features["label"].names
ds_nd = ds.cast_column("image", Image(decode=False))

def parse_subject_and_finger(path: str):
    m = re.search(r"Captured images/(\d{3})/([^/]+)/", path)
    if not m:
        return None, None
    return m.group(1), m.group(2)

def add_finger_key(ex):
    path = ex["image"]["path"]
    subject, finger = parse_subject_and_finger(path)
    if subject is None:
        finger = label_names[ex["label"]]
        return {"finger_key": f"UNKNOWN_{finger}"}
    return {"finger_key": f"{subject}_{finger}"}

ds2 = ds_nd.map(add_finger_key)
uniq_keys = sorted(set(ds2["finger_key"]))
print("Unique finger identities (subject×finger):", len(uniq_keys))
print("First 10 keys:", uniq_keys[:10])
assert len(uniq_keys) > 50, "Too few identities; parsing failed."

key2id = {k:i for i,k in enumerate(uniq_keys)}
ds3 = ds2.map(lambda ex: {"finger_id": key2id[ex["finger_key"]]})
ds3 = ds3.cast_column("image", Image(decode=True))  # back to PIL

# -----------------------
# Preprocessing (same as before)
# -----------------------
def crop_largest_contour(img):
    blur = cv2.GaussianBlur(img, (5,5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((15,15), np.uint8)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return img
    cnt = max(contours, key=cv2.contourArea)
    x,y,w,h = cv2.boundingRect(cnt)

    H, W = img.shape
    area = w*h
    if area < 0.10*H*W or area > 0.98*H*W:
        s = int(min(H,W)*0.85)
        y1 = (H - s)//2; x1 = (W - s)//2
        return img[y1:y1+s, x1:x1+s]

    margin = 10
    x1 = max(0, x - margin)
    y1 = max(0, y - margin)
    x2 = min(W, x + w + margin)
    y2 = min(H, y + h + margin)
    return img[y1:y2, x1:x2]

def apply_clahe(img, clipLimit=2.0, tileGridSize=(8,8)):
    clahe = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=tileGridSize)
    return clahe.apply(img)

def resize_with_pad(img, size=224):
    h, w = img.shape
    scale = size / max(h, w)
    new_w = max(1, int(w*scale))
    new_h = max(1, int(h*scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad = np.zeros((size, size), dtype=np.uint8)
    y = (size - new_h) // 2
    x = (size - new_w) // 2
    pad[y:y+new_h, x:x+new_w] = resized
    return pad

def preprocess(pil_img, size=224):
    img = np.array(pil_img).astype(np.uint8)
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    if use_preprocess:
        img = crop_largest_contour(img)
        img = cv2.medianBlur(img, 3)
        img = apply_clahe(img, clipLimit=2.0, tileGridSize=(8,8))
    img = resize_with_pad(img, size)
    img = img.astype(np.float32) / 255.0
    return img

# -----------------------
# Split (same logic)
# -----------------------
finger_ids = np.array(ds3["finger_id"])
idx_by_id = defaultdict(list)
for i, fid in enumerate(finger_ids):
    idx_by_id[int(fid)].append(i)

train_idx, val_idx = [], []
rng = np.random.default_rng(seed)
for fid, idxs in idx_by_id.items():
    idxs = np.array(idxs)
    rng.shuffle(idxs)
    cut = max(1, int(0.8 * len(idxs)))
    train_idx.extend(idxs[:cut].tolist())
    val_idx.extend(idxs[cut:].tolist())

print("Train samples:", len(train_idx), "Val samples:", len(val_idx))

class VeinDS(Dataset):
    def __init__(self, hf_ds, indices):
        self.ds = hf_ds
        self.idx = indices
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        item = self.ds[int(self.idx[i])]
        x = preprocess(item["image"], size=img_size)
        x = torch.from_numpy(x).unsqueeze(0)
        y = int(item["finger_id"])
        return x, y

train_loader = DataLoader(VeinDS(ds3, train_idx), batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(VeinDS(ds3, val_idx), batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)

num_ids = len(uniq_keys)

# -----------------------
# Model: ResNet34 backbone
# -----------------------
class EmbedNet(nn.Module):
    def __init__(self, num_classes, emb_dim=256):
        super().__init__()
        base = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        w = base.conv1.weight.data
        base.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        base.conv1.weight.data = w.mean(dim=1, keepdim=True)

        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.emb = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, emb_dim),
            nn.BatchNorm1d(emb_dim),
        )
        self.cls = nn.Linear(emb_dim, num_classes)

    def forward(self, x):
        z = self.backbone(x)
        e = self.emb(z)
        e = F.normalize(e, dim=1)
        logits = self.cls(e)
        return logits, e

model = EmbedNet(num_classes=num_ids, emb_dim=emb_dim).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)
criterion = nn.CrossEntropyLoss()

# -----------------------
# Resume
# -----------------------
start_epoch = 1
best_eer = 1e9
if os.path.exists("last_checkpoint.pt"):
    ckpt = torch.load("last_checkpoint.pt", map_location=device)
    model.load_state_dict(ckpt["model"])
    opt.load_state_dict(ckpt["opt"])
    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch = ckpt["epoch"] + 1
    best_eer = ckpt.get("best_eer", 1e9)
    print(f"✅ Resumed from epoch {ckpt['epoch']} | best_eer={best_eer*100:.2f}% | LR={opt.param_groups[0]['lr']}")
else:
    print("Starting fresh training.")

# -----------------------
# EER helpers
# -----------------------
def compute_eer(scores, labels01):
    order = np.argsort(-scores)
    scores = scores[order]
    labels01 = labels01[order]
    P = labels01.sum()
    N = len(labels01) - P
    tp = fp = 0
    fn = P
    tn = N
    best = 1e9
    eer = None
    thr = None
    for i in range(len(scores)):
        if labels01[i] == 1:
            tp += 1; fn -= 1
        else:
            fp += 1; tn -= 1
        far = fp / (fp + tn + 1e-12)
        frr = fn / (fn + tp + 1e-12)
        d = abs(far - frr)
        if d < best:
            best = d
            eer = (far + frr) / 2
            thr = scores[i]
    return float(eer), float(thr)

@torch.no_grad()
def extract_embeddings(loader):
    model.eval()
    embs, labs = [], []
    for x,y in tqdm(loader, desc="Embed", leave=False):
        x = x.to(device, non_blocking=True)
        _, e = model(x)
        embs.append(e.cpu())
        labs.append(y)
    return torch.cat(embs, 0).numpy(), torch.cat(labs, 0).numpy()

def verification_eer(embs, labs, max_pairs=40000):
    rng = np.random.default_rng(seed)
    idx_by = defaultdict(list)
    for i, lb in enumerate(labs):
        idx_by[int(lb)].append(i)
    ids = list(idx_by.keys())

    scores, labels01 = [], []

    # genuine
    for lb, idxs in idx_by.items():
        if len(idxs) < 2:
            continue
        k = min(30, len(idxs)*(len(idxs)-1)//2)
        for _ in range(k):
            a,b = rng.choice(idxs, 2, replace=False)
            scores.append(float(np.dot(embs[a], embs[b])))
            labels01.append(1)
            if len(labels01) >= max_pairs//2:
                break
        if len(labels01) >= max_pairs//2:
            break

    # impostor
    while len(labels01) < max_pairs:
        la, lb = rng.choice(ids, 2, replace=False)
        a = rng.choice(idx_by[la])
        b = rng.choice(idx_by[lb])
        scores.append(float(np.dot(embs[a], embs[b])))
        labels01.append(0)

    return compute_eer(np.array(scores, np.float32), np.array(labels01, np.int32))

# -----------------------
# Train + Save to Drive every epoch
# -----------------------
for ep in range(start_epoch, epochs+1):
    model.train()
    total_loss, total, correct = 0.0, 0, 0

    for x,y in tqdm(train_loader, desc=f"Train {ep}/{epochs}", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        logits, _ = model(x)
        loss = criterion(logits, y)
        loss.backward()
        opt.step()

        total_loss += loss.item() * x.size(0)
        total += x.size(0)
        correct += (logits.argmax(1) == y).sum().item()

    tr_loss = total_loss / total
    tr_acc = correct / total

    embs, labs = extract_embeddings(val_loader)
    eer, thr = verification_eer(embs, labs, max_pairs=max_pairs)

    print(f"Epoch {ep}/{epochs} | train loss={tr_loss:.4f} acc={tr_acc*100:.2f}% | Val EER={eer*100:.2f}% @ thr={thr:.4f}")

    scheduler.step()
    print("  LR now:", opt.param_groups[0]["lr"])

    save_obj = {
        "epoch": ep,
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_eer": best_eer,
        "settings": {"epochs": epochs, "batch_size": batch_size, "lr": lr, "emb_dim": emb_dim, "img_size": img_size}
    }
    torch.save(save_obj, "last_checkpoint.pt")
    shutil.copy("last_checkpoint.pt", DRIVE_LAST)
    print("  💾 saved last_checkpoint.pt (Drive)")

    if eer < best_eer:
        best_eer = eer
        save_obj["best_eer"] = best_eer
        torch.save(save_obj, "best_checkpoint.pt")
        shutil.copy("best_checkpoint.pt", DRIVE_BEST)
        print(f"  ⭐ saved best_checkpoint.pt (Drive) new best EER = {best_eer*100:.2f}%")

print("Done. Best EER so far:", best_eer*100, "%")
print("Drive folder:", DRIVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

MMCBNU_6000.zip:   0%|          | 0.00/677M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12000 [00:00<?, ? examples/s]

...

Epoch 4/10 | train loss=4.6766 acc=83.78% | Val EER=1.15% @ thr=0.6450
  LR now: 0.0003
  💾 saved last_checkpoint.pt (Drive)
  ⭐ saved best_checkpoint.pt (Drive) new best EER = 1.15%


Epoch 5/10 | train loss=4.2682 acc=85.92% | Val EER=1.79% @ thr=0.6664
  LR now: 0.00015
  💾 saved last_checkpoint.pt (Drive)


Epoch 6/10 | train loss=3.9348 acc=91.77% | Val EER=1.15% @ thr=0.5882
  LR now: 0.00015
  💾 saved last_checkpoint.pt (Drive)


Epoch 7/10 | train loss=3.7466 acc=93.78% | Val EER=0.78% @ thr=0.5815
  LR now: 0.00015
  💾 saved last_checkpoint.pt (Drive)
  ⭐ saved best_checkpoint.pt (Drive) new best EER = 0.78%


KeyboardInterrupt: 